# Component rotation measures with Python RM synthesis
This notebook implements the instrumental-response method in [PSRSALSA commit 2efbaf4](https://github.com/weltevrede/psrsalsa/tree/2efbaf4592e08fdb76002bbc9dc25e8e4441813b) using Python and NumPy. PSRCHIVE reads the observation. The synthetic check runs independently of telescope data or a PSRSALSA installation.

In [ ]:
from pathlib import Path
import csv
import json
import tempfile
import numpy as np
import matplotlib.pyplot as plt

INPUT = Path("/fred/oz341/B1259_kavya/parkes/high_res/test_archives_rfi/sub/ext")
EXTENSION = ".center"
OUTPUT = Path("python_psrsalsa_rm")
RUN_ARCHIVES = True
RM_GRID = (-10000., 10000., 20000)  # inclusive endpoints, sample count
FTOL = 0.001
BOOTSTRAP = 0  # use e.g. 500 to fit Gaussian noise realisations
SEED = 42
LONGITUDE_FITS = False
PEAK_SIGMA = 3.0
SMOOTH_PHASE = 0.004

## Identify the components
The weighted Stokes I profile is smoothed across phase zero. Significant peaks and the valley between them define up to two gates. Their union defines the on-pulse window used in the noise and analytic calculations. Supply baseline-subtracted, single-subintegration archives before Faraday correction.

In [ ]:
def detect(profile):
    n = len(profile)
    width = max(0.7, n * SMOOTH_PHASE)
    radius = max(1, round(4*width))
    kernel = np.exp(-0.5*(np.arange(-radius, radius+1)/width)**2)
    smooth = np.convolve(np.pad(profile, radius, mode="wrap"), kernel/kernel.sum(), mode="valid")
    origin = int(smooth.argmin())
    y = np.roll(smooth, -origin)
    baseline = np.median(y)
    noise = max(1.4826 * np.median(abs(y - baseline)), np.finfo(float).tiny)
    candidates = np.flatnonzero((y > np.roll(y, 1)) & (y >= np.roll(y, -1))
                                & (y > baseline + PEAK_SIGMA*noise))
    selected = []
    for peak in candidates[np.argsort(y[candidates])[::-1]]:
        if not selected:
            selected.append(peak)
            continue
        a, b = sorted([selected[0], peak])
        # A separate component needs a resolved valley on both circular paths.
        saddle = max(y[a:b+1].min(), min(y[:a+1].min(), y[b:].min()))
        if min(b-a, n-b+a) >= max(2, round(3*width)) and y[peak]-saddle >= PEAK_SIGMA*noise:
            selected.append(peak)
            break
    chosen = sorted(selected)
    gates = []
    for p in chosen:
        left, right = p, p
        while left > 0 and y[left-1] > baseline + 2*noise:
            left -= 1
        while right < n-1 and y[right+1] > baseline + 2*noise:
            right += 1
        if len(chosen) == 2:
            a, b = chosen
            valley = a + np.argmin(y[a:b+1])
            if p == a:
                right = min(right, valley)
            else:
                left = max(left, valley+1)
        gates.append(np.roll((np.arange(n) >= left) & (np.arange(n) <= right), origin))
    gates.sort(key=lambda gate: np.argmax(np.where(gate, smooth, -np.inf)))
    return gates

In [ ]:
def read_archive(path):
    import psrchive
    archive = psrchive.Archive_load(str(path))
    if archive.get_npol() != 4 or archive.get_nsubint() != 1:
        raise ValueError("Prepare a single-subintegration, full-Stokes archive for rmsynth")
    if archive.get_faraday_corrected():
        raise ValueError("Use an archive before Faraday correction")
    archive.convert_state("Stokes")
    archive.dedisperse()
    cube = archive.get_data()[0].astype(float)
    data = cube[0]
    weights = archive.get_weights()[0]
    good = (weights > 0) & np.isfinite(weights) & np.isfinite(cube).all(axis=(0, 2))
    if not good.any():
        raise ValueError("No usable intensity channels")
    profile = np.average(data[good], axis=0, weights=weights[good])
    cube[:, ~good] = 0.
    integration = archive.get_Integration(0)
    freq = np.array([integration.get_centre_frequency(k) for k in range(len(weights))])
    step = archive.get_bandwidth()/len(freq)
    nominal = archive.get_centre_frequency() + (np.arange(len(freq))-(len(freq)-1)/2)*step
    if len(freq)<2 or step == 0 or np.any(~np.isfinite(nominal)) or np.any(nominal <= 0):
        raise ValueError("Invalid nominal channel geometry")
    if np.any(~np.isfinite(freq[good])) or np.any(freq[good] <= 0):
        raise ValueError("Invalid frequency on a usable channel")
    freq[~good] = nominal[~good]
    return profile, freq, nominal, cube, good

def intervals(mask):
    edges = np.diff(np.r_[False, mask, False].astype(int))
    return list(zip(np.flatnonzero(edges == 1).tolist(),
                    (np.flatnonzero(edges == -1) - 1).tolist()))

## Fit a shifted instrumental response
For each trial RM, rotate the channel vectors Q+iU and sum them separately at each pulse longitude. Take the absolute value at each longitude and sum those amplitudes across the gate. Fit scale × response + offset to that spectrum across the entire grid. The response is the modulus of the mean phasor across nominal, uniformly spaced channel centres. The simplex starts at the grid midpoint and uses the upstream fractional objective tolerance. Inspect the fitted RM alongside the sampled maximum on broad grids.

In [ ]:
def simplex(loss, start, step):
    # Keep the vertex order, tie handling and shrink updates of amoeba_nmsimplex_d.
    vertices = np.vstack([start, start + np.diag(step)])
    values = np.array([loss(v) for v in vertices])
    for _ in range(5000):
        best, worst = int(values.argmin()), int(values.argmax())
        second = best
        for i, value in enumerate(values):
            if values[second] < value < values[worst]:
                second = i
        center = np.delete(vertices, worst, axis=0).mean(axis=0)
        reflected = 2*center - vertices[worst]
        trial = loss(reflected)
        if values[best] <= trial < values[second]:
            vertices[worst], values[worst] = reflected, trial
        if trial < values[best]:
            expanded = center + 2*(reflected-center)
            expanded_cost = loss(expanded)
            vertices[worst], values[worst] = ((expanded, expanded_cost)
                                               if expanded_cost < trial else (reflected, trial))
        if trial >= values[second]:
            end = reflected if trial < values[worst] else vertices[worst]
            contracted = center + .5*(end-center)
            cost = loss(contracted)
            if cost < values[worst]:
                vertices[worst], values[worst] = contracted, cost
            else:
                vertices[:] = vertices[best] + .5*(vertices-vertices[best])
                for i in (worst, second):
                    values[i] = loss(vertices[i])
        low, high = values.min(), values.max()
        if high == 0 or 2*(high-low) < FTOL*(abs(high)+abs(low)):
            return vertices[values.argmin()].copy()
    raise RuntimeError("Instrumental-response fit exceeded 5000 iterations")

def response_function(grid, nominal):
    l2 = (299.792458 / nominal)**2
    # Reuse the double-precision response matrix when it fits a 128 MiB budget.
    if grid.size * l2.size * 16 <= 128 * 1024**2:
        bank = np.exp(-2j * grid[:, None] * l2)
        def response(shift):
            return abs(bank @ np.exp(2j*shift*l2))/len(l2)
    else:
        def response(shift):
            values = np.empty(len(grid))
            for i in range(0, len(grid), 128):
                trial = grid[i:i+128] - shift
                values[i:i+128] = abs(np.exp(-2j*trial[:,None]*l2).mean(axis=1))
            return values
    return response

def fit_response(grid, spectrum, response):
    def loss(params):
        shift, scale, offset = params
        residual = scale*response(shift) + offset - spectrum
        return float(residual @ residual)
    peak = float(spectrum.max())
    if peak <= 0 or not np.isfinite(spectrum).all():
        raise ValueError("Cannot fit an empty or nonfinite spectrum")
    start = np.array([(grid[0]+grid[-1])/2, peak, 0.])
    step = .05*np.array([np.ptp(grid), peak, peak])
    params = simplex(loss, start, step)
    shift, scale, offset = params
    return params, scale*response(shift)+offset

## Uncertainties
For the analytic error, derotate at the fitted RM, sum channels, debias linear polarisation and measure its integrated signal-to-noise ratio. The width is the reciprocal sample standard deviation of nominal wavelength squared. The collapsed uncertainty is width / (2 × S/N). Longitude fits use the mean bin RM for derotation and share the upstream longitude-mode analytic error. The optional bootstrap adds independent Gaussian noise to Q and U with each channel’s off-pulse Stokes I population standard deviation, repeats the fit, and reports the population spread of fitted RMs.

In [ ]:
def mask_regions(pairs, n):
    bins = np.arange(n)
    mask = np.zeros(n, bool)
    for left, right in pairs:
        if not (0 <= left < n and 0 <= right < n):
            raise ValueError(f"Region {(left,right)} outside 0..{n-1}")
        mask |= ((bins >= left) & (bins <= right) if left <= right
                 else (bins >= left) | (bins <= right))
    if not mask.any():
        raise ValueError("Empty region")
    return mask

def analytic_error(pol, freq, nominal, on2, rm, longitude=False):
    derotated = np.exp(-2j * rm * (299.792458/freq)**2) @ pol
    raw_l2 = abs(derotated)**2
    rms_l = np.sqrt(raw_l2[~on2].mean())
    debiased = np.sqrt(np.maximum(raw_l2 - rms_l**2/2, 0))
    total = debiased[on2].sum()
    count = on2.sum()
    sn = total/(np.sqrt(count)*rms_l) if rms_l > 0 else np.nan
    if longitude:
        sn /= count
    width = 1/np.std((299.792458/nominal)**2, ddof=1)
    return dict(sigma_analytic=width/(2*sn) if sn > 0 else np.nan,
                fwhm=width, sn_l=sn, rms_l=rms_l, sum_l=total)

def compare_cube(freq, nominal, cube, gate, on2, settings, draws, long_mode, seed):
    low, high, count = settings
    if low >= high or count < 3 or draws < 0 or draws == 1 or FTOL <= 0:
        raise ValueError("Check RM grid, bootstrap count and fit tolerance")
    if not gate.any() or not on2.any() or (~on2).sum() < 2:
        raise ValueError("Need signal bins and at least two noise bins")
    grid = np.linspace(low, high, count)
    pol = cube[1] + 1j*cube[2]
    signal = pol[:, gate]
    rms_i = cube[0][:, ~on2].std(axis=1, ddof=0)[:, None]
    wavelength = (299.792458/freq).astype(np.float32)
    response = response_function(grid, nominal)
    rng = np.random.default_rng(seed)
    bins = np.flatnonzero(gate)
    estimates, rows = [], []
    for iteration in range(draws+1):
        sample = signal
        if iteration:
            noise = rng.normal(size=signal.shape) + 1j*rng.normal(size=signal.shape)
            sample = signal + rms_i*noise
        sample = sample.astype(np.complex64)
        amps = np.empty((len(grid), len(bins)), np.float32)
        for i in range(0, len(grid), 128):
            phase = -2*grid[i:i+128,None].astype(np.float32)*(wavelength*wavelength)
            amps[i:i+128] = abs(np.exp(1j*phase) @ sample)
        collapsed = amps.sum(axis=1)
        targets = [collapsed] + (list(amps.T) if long_mode else [])
        fits = [fit_response(grid, y, response) for y in targets]
        rmlist = [params[0] for params, curve in fits]
        if iteration:
            estimates.append(rmlist)
            continue
        products = dict(grid=grid, bins=bins, amplitudes=amps, collapsed=collapsed,
                        fitted_curve=fits[0][1], onpulse2=np.flatnonzero(on2),
                        freq_mhz=freq, nominal_mhz=nominal, channel_sigma_i=rms_i[:,0])
        for j, ((rm, scale, offset), _) in enumerate(fits):
            err = analytic_error(pol, freq, nominal, on2,
                                 np.mean(rmlist[1:]) if j else rm, bool(j))
            rows.append(dict(bin=int(bins[j-1]) if j else "collapsed", rm=rm,
                             scale=scale, offset=offset, grid_peak=grid[targets[j].argmax()],
                             outside_grid=not low <= rm <= high, **err))
    samples = np.asarray(estimates).reshape(draws,len(rows))
    products["bootstrap_rm"] = samples
    for j,row in enumerate(rows):
        row["sigma_bootstrap"] = samples[:,j].std(ddof=0) if draws else np.nan
    return rows, products

## Synthetic check
Two equal components in adjacent bins have opposite polarisation vectors. The collapsed amplitude should peak at 24 and the response fit should recover the injected +7.5 rad/m². The check also repeats the bootstrap with a fixed seed.

In [ ]:
def synthetic_check():
    freq = np.linspace(900., 1500., 12)
    cube = np.random.default_rng(19).normal(0,.02,(4,12,32))
    truth = 7.5
    phasor = np.exp(2j*truth*(299.792458/freq)**2)
    cube[1,:,8], cube[2,:,8] = phasor.real, phasor.imag
    cube[1,:,9], cube[2,:,9] = -phasor.real, -phasor.imag
    gate = mask_regions([(8,9)],32)
    args = (freq,freq,cube,gate,gate,(-30.,30.,121),4,True,31)
    rows, product = compare_cube(*args)
    expected = [2*abs(np.exp(2j*(truth-r)*(299.792458/freq)**2).sum()) for r in product["grid"]]
    np.testing.assert_allclose(product["collapsed"], expected, atol=1e-5)
    np.testing.assert_allclose(product["fitted_curve"], expected, atol=.02)
    repeat = compare_cube(*args)[1]
    np.testing.assert_array_equal(product["bootstrap_rm"], repeat["bootstrap_rm"])
    np.testing.assert_allclose(product["collapsed"].max(),24,rtol=1e-6)
    np.testing.assert_allclose((cube[1]+1j*cube[2])[:,gate].sum(axis=1),0)
    np.testing.assert_allclose([r["rm"] for r in rows],truth,atol=.01)
    np.testing.assert_allclose(rows[0]["sigma_bootstrap"],product["bootstrap_rm"][:,0].std())
    assert rows[0]["sigma_analytic"] > 0 and rows[0]["sigma_bootstrap"] > 0
    print(f"Injected {truth:.3f}, recovered {rows[0]['rm']:.5f} rad/m²; all checks passed.")
synthetic_check()

## Analyse the archives
Inspect gates.png, the RM-versus-bin map and the fitted curve for each component. A fresh output directory keeps runs separate.

In [ ]:
def analyse(path, root):
    folder = root / path.name
    folder.mkdir()
    profile, freq, nominal, cube, good = read_archive(path)
    gates = detect(profile)
    labels = ["single"] if len(gates) == 1 else ["leading", "trailing"]
    regions = {name: intervals(gate) for name, gate in zip(labels, gates)}
    (folder / "gates.json").write_text(json.dumps(regions, indent=2))
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(profile, color="black", lw=1)
    for name, ranges in regions.items():
        for i, (a, b) in enumerate(ranges):
            ax.axvspan(a-.5, b+.5, alpha=.25, color="C0" if name != "trailing" else "C1",
                       label=name if i == 0 else None)
    if gates:
        ax.legend()
    ax.set(xlabel="Archive phase bin", ylabel="Stokes I", title=path.name)
    fig.tight_layout()
    fig.savefig(folder / "gates.png")
    plt.show()
    plt.close(fig)
    if not gates:
        print(path.name, "No significant component; see gates.png")
        return
    signal = np.logical_or.reduce(gates)
    if signal.all():
        raise ValueError("No off-pulse bins remain for native uncertainty estimates")
    for name, gate in zip(labels, gates):
        print(path.name, name, "fitting", RM_GRID[2], "RM samples", flush=True)
        rows, result = compare_cube(freq, nominal, cube, gate, signal, RM_GRID,
                                    BOOTSTRAP, LONGITUDE_FITS, SEED)
        np.savez_compressed(folder / (name+".npz"), **result, valid_channels=good)
        with (folder / (name+".csv")).open("w", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=rows[0])
            writer.writeheader()
            writer.writerows(rows)
        grid = result["grid"]
        fig, (a,b) = plt.subplots(1,2,figsize=(10,4), constrained_layout=True)
        a.imshow(result["amplitudes"], aspect="auto", origin="lower",
                 extent=(-.5,gate.sum()-.5,grid[0],grid[-1]))
        a.set(xlabel="Index in selected phase bins", ylabel="RM (rad/m²)")
        b.plot(grid,result["collapsed"],label="Collapsed spectrum")
        b.plot(grid,result["fitted_curve"],"--",label="Response fit")
        b.set(xlabel="RM (rad/m²)", ylabel="Polarised amplitude")
        b.legend()
        fig.suptitle(f"{path.name} · {name}")
        fig.savefig(folder / (name+".png"))
        plt.show()
        plt.close(fig)
        print(rows[0])

In [ ]:
if RUN_ARCHIVES:
    files = [INPUT] if INPUT.is_file() else sorted(p for p in INPUT.glob("*"+EXTENSION) if p.is_file())
    if not files:
        raise FileNotFoundError(f"No matching archives at {INPUT}")
    OUTPUT.mkdir(parents=True, exist_ok=True)
    root = Path(tempfile.mkdtemp(prefix="run-", dir=OUTPUT)).resolve()
    print("Results:", root)
    (root / "settings.json").write_text(json.dumps(dict(grid=RM_GRID, ftol=FTOL,
        bootstrap=BOOTSTRAP, seed=SEED, longitude=LONGITUDE_FITS, peak_sigma=PEAK_SIGMA,
        smooth_phase=SMOOTH_PHASE, numpy=np.__version__), indent=2))
    errors = {}
    for path in files:
        try:
            analyse(path, root)
        except (OSError, ValueError, RuntimeError) as exc:
            errors[str(path)] = str(exc)
            print(path.name, exc)
    if errors:
        (root / "errors.json").write_text(json.dumps(errors, indent=2))
        raise RuntimeError(f"{len(errors)} failed archive(s); see {root / 'errors.json'}")

## Implementation references
See the pinned [RM synthesis and fit](https://github.com/weltevrede/psrsalsa/blob/2efbaf4592e08fdb76002bbc9dc25e8e4441813b/src/lib/rmsynth.c) and [program and uncertainty calculation](https://github.com/weltevrede/psrsalsa/blob/2efbaf4592e08fdb76002bbc9dc25e8e4441813b/src/prog/rmsynth.c). The simplex follows the Hutt implementation modified by Patrick Weltevrede.

Copyright (c) 1997-2011 <Michael F. Hutt>

 Permission is hereby granted, free of charge, to any person obtaining
 a copy of this software and associated documentation files (the
 "Software"), to deal in the Software without restriction, including
 without limitation the rights to use, copy, modify, merge, publish,
 distribute, sublicense, and/or sell copies of the Software, and to
 permit persons to whom the Software is furnished to do so, subject to
 the following conditions:

 The above copyright notice and this permission notice shall be
 included in all copies or substantial portions of the Software.

 THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,
 EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
 MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND
 NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE
 LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION
 OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION
 WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.